# 8. Generate Realistic Slow Queries and Metrics

## 8.1 Import Libraries and Setup

In [1]:
import mysql.connector
import pandas as pd
import random
import time
import psutil
import os
import threading

In [2]:
DB_CFG = dict(
    host="127.0.0.1",
    port=3307,
    user="sadop_user",
    password="1234",
    database="SADOP_BDD"
)
conn   = mysql.connector.connect(**DB_CFG)
cursor = conn.cursor()


In [3]:
kill_conn   = mysql.connector.connect(**DB_CFG)
kill_cursor = kill_conn.cursor()

print("Connected to SADOP database")

Connected to SADOP database


In [4]:
QUERY_TIMEOUT = 10

In [5]:
def get_explain_metrics(sql: str) -> dict:
    """
    Returns a dict with:
        estimated_rows  – total rows MySQL estimates it will examine
        uses_index      – 1 if at least one step uses a key/index
        full_table_scan – 1 if at least one step is a full scan (type='ALL')
        uses_filesort   – 1 if 'Using filesort'  appears in any Extra
        uses_temp_table – 1 if 'Using temporary' appears in any Extra
    Returns all-zero dict on any error.
    """
    defaults = dict(
        estimated_rows=None,
        uses_index=0,
        full_table_scan=0,
        uses_filesort=0,
        uses_temp_table=0,
    )
    try:
        exp_cursor = conn.cursor()
        exp_cursor.execute(f"EXPLAIN {sql}")
        rows = exp_cursor.fetchall()
        cols = [d[0].lower() for d in exp_cursor.description]
        exp_cursor.close()

        estimated_rows  = 0
        uses_index      = 0
        full_table_scan = 0
        uses_filesort   = 0
        uses_temp_table = 0

        for row in rows:
            r = dict(zip(cols, row))

            # estimated_rows  → EXPLAIN column "rows"
            if r.get("rows") is not None:
                try:
                    estimated_rows += int(r["rows"])
                except (ValueError, TypeError):
                    pass

            # uses_index  → EXPLAIN column "key" is not NULL / empty
            key_val = r.get("key") or r.get("possible_keys")
            if key_val:
                uses_index = 1

            # full_table_scan → type == 'ALL'
            if str(r.get("type", "")).upper() == "ALL":
                full_table_scan = 1

            # filesort / temp table → Extra column
            extra = str(r.get("extra", "")).lower()
            if "using filesort"   in extra:
                uses_filesort   = 1
            if "using temporary"  in extra:
                uses_temp_table = 1

        return dict(
            estimated_rows=estimated_rows if estimated_rows > 0 else None,
            uses_index=uses_index,
            full_table_scan=full_table_scan,
            uses_filesort=uses_filesort,
            uses_temp_table=uses_temp_table,
        )

    except Exception as e:
        print(f"  [EXPLAIN error] {e}")
        return defaults


In [6]:
def execute_with_timeout(sql: str, timeout: int):
    """
    Runs *sql* on the shared cursor.
    If it takes longer than *timeout* seconds the query is killed via
    a separate connection and (rows=[], timed_out=True) is returned.
    """
    result       = {"rows": [], "timed_out": False, "error": None}
    timer        = None

    def _kill():
        try:
            # Fetch the processlist id of our running query
            kill_cursor.execute(
                "SELECT id FROM information_schema.processlist "
                "WHERE user = %s AND command = 'Query' "
                "ORDER BY time DESC LIMIT 1",
                (DB_CFG["user"],)
            )
            row = kill_cursor.fetchone()
            if row:
                kill_cursor.execute(f"KILL QUERY {row[0]}")
                kill_conn.commit()
                print(f"  [TIMEOUT] Query killed after {timeout}s")
        except Exception as ke:
            print(f"  [KILL error] {ke}")
        result["timed_out"] = True

    try:
        timer = threading.Timer(timeout, _kill)
        timer.start()

        cursor.execute(sql)
        result["rows"] = cursor.fetchall()

    except mysql.connector.Error as e:
        # MySQL error 1317 = "Query execution was interrupted"
        if e.errno == 1317 or result["timed_out"]:
            result["timed_out"] = True
            # Drain any pending result to keep the cursor clean
            try:
                cursor.fetchall()
            except Exception:
                pass
        else:
            result["error"] = str(e)
    finally:
        if timer:
            timer.cancel()

    return result

In [7]:
QUERY_TEMPLATES = [

# ===============================
# BASIC JOIN + WHERE
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, u.full_name, t.amount
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
WHERE u.user_id = {uid}
""",

# ===============================
# JOIN + ORDER BY (filesort)
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, t.transaction_date, t.amount
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
ORDER BY t.transaction_date DESC
""",

# ===============================
# GROUP BY + HAVING
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, SUM(t.amount) AS total_amount
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
GROUP BY u.user_id
HAVING total_amount > {amt}
""",

# ===============================
# GROUP BY account
# ===============================
lambda uid, amt: f"""
SELECT a.account_id, COUNT(*) AS tx_count
FROM accounts a
JOIN transactions t ON a.account_id = t.account_id
GROUP BY a.account_id
HAVING tx_count > 5
""",

# ===============================
# IN subquery (slow)
# ===============================
lambda uid, amt: f"""
SELECT *
FROM user
WHERE user_id IN (
    SELECT user_id
    FROM accounts
    WHERE account_id IN (
        SELECT account_id
        FROM transactions
        WHERE amount > {amt}
    )
)
""",

# ===============================
# EXISTS
# ===============================
lambda uid, amt: f"""
SELECT *
FROM user u
WHERE EXISTS (
    SELECT 1
    FROM accounts a
    JOIN transactions t ON a.account_id = t.account_id
    WHERE a.user_id = u.user_id
      AND t.amount > {amt}
)
""",

# ===============================
# Correlated subquery
# ===============================
lambda uid, amt: f"""
SELECT u.user_id,
       (
         SELECT SUM(t.amount)
         FROM accounts a
         JOIN transactions t ON a.account_id = t.account_id
         WHERE a.user_id = u.user_id
       ) AS total_amount
FROM user u
""",

# ===============================
# COUNT with WHERE
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, COUNT(t.transaction_id) AS tx_count
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
WHERE t.amount > {amt}
GROUP BY u.user_id
""",

# ===============================
# ORDER BY SUM
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, SUM(t.amount) AS total_amount
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
GROUP BY u.user_id
ORDER BY total_amount DESC
""",

# ===============================
# JOIN logs (4-table join)
# ===============================
lambda uid, amt: f"""
SELECT u.user_id, l.log_level, COUNT(*) AS log_count
FROM user u
JOIN logs l ON u.user_id = l.user_id
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
GROUP BY u.user_id, l.log_level
""",

# ===============================
# DISTINCT + JOIN
# ===============================
lambda uid, amt: f"""
SELECT DISTINCT u.user_id
FROM user u
JOIN accounts a ON u.user_id = a.user_id
JOIN transactions t ON a.account_id = t.account_id
WHERE t.amount > {amt}
""",

# ===============================
# LIKE (non-SARGable)
# ===============================
lambda uid, amt: f"""
SELECT *
FROM user
WHERE email LIKE '%gmail%'
""",

# ===============================
# OR condition (index killer)
# ===============================
lambda uid, amt: f"""
SELECT *
FROM transactions
WHERE amount > {amt}
   OR transaction_date < '2025-05-02'
""",

# ===============================
# UNION
# ===============================
lambda uid, amt: f"""
SELECT user_id FROM user
UNION
SELECT user_id FROM accounts
""",

# ===============================
# LEFT JOIN + IS NULL
# ===============================
lambda uid, amt: f"""
SELECT u.user_id
FROM user u
LEFT JOIN accounts a ON u.user_id = a.user_id
WHERE a.account_id IS NULL
""",

# ===============================
# Nested aggregation
# ===============================
lambda uid, amt: f"""
SELECT AVG(total_amount)
FROM (
    SELECT SUM(t.amount) AS total_amount
    FROM accounts a
    JOIN transactions t ON a.account_id = t.account_id
    GROUP BY a.user_id
) sub
""",
]


In [8]:
cursor.execute("SELECT COUNT(*) FROM user")
count = cursor.fetchone()[0]
print("Users in DB:", count)

Users in DB: 20000


## 8.2 Simulate Heavy Queries with Joins and Aggregation


In [10]:
metrics = []
NUM_QUERIES = 20000

cursor.execute("SELECT user_id FROM user")
all_user_ids = [row[0] for row in cursor.fetchall()]

cursor.execute("SELECT MIN(amount), MAX(amount) FROM transactions")
min_amt, max_amt = cursor.fetchone()

print(f"Running {NUM_QUERIES} queries (timeout = {QUERY_TIMEOUT}s each) …")

for i in range(NUM_QUERIES):
    user_id          = random.choice(all_user_ids)
    amount_threshold = random.randint(int(min_amt), int(max_amt))
    query_sql        = random.choice(QUERY_TEMPLATES)(user_id, amount_threshold)
    clean_sql        = query_sql.strip()

    # ── EXPLAIN metrics (always collected, fast) ──
    explain = get_explain_metrics(clean_sql)

    # ── Execute with timeout ──
    start_time = time.time()
    outcome    = execute_with_timeout(clean_sql, QUERY_TIMEOUT)
    exec_time  = time.time() - start_time

    if i % 500 == 0:
        status = "TIMEOUT" if outcome["timed_out"] else f"{exec_time:.3f}s"
        print(f"  [{i}/{NUM_ vQUERIES}] {status}")

    metrics.append({
        "query":            clean_sql,
        "query_time":       exec_time,
        "timed_out":        int(outcome["timed_out"]),
        "rows_returned":    len(outcome["rows"]),
        "has_sum":          int("SUM"      in query_sql.upper()),
        "has_group_by":     int("GROUP BY" in query_sql.upper()),
        "has_where":        int("WHERE"    in query_sql.upper()),
        "tables_count":     query_sql.lower().count("join") + 1,
        "query_length":     len(query_sql),
        "cpu_usage":        psutil.cpu_percent(interval=0.01),
        # ── EXPLAIN features ──
        "estimated_rows":   explain["estimated_rows"],
        "uses_index":       explain["uses_index"],
        "full_table_scan":  explain["full_table_scan"],
        "uses_filesort":    explain["uses_filesort"],
        "uses_temp_table":  explain["uses_temp_table"],
    })


Running 20000 queries (timeout = 10s each) …
  [0/20000] 0.851s
  [TIMEOUT] Query killed after 10s
  [500/20000] 0.014s
  [1000/20000] 0.053s
  [TIMEOUT] Query killed after 10s
  [1500/20000] 0.052s
  [2000/20000] 0.560s
  [2500/20000] 0.186s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [3000/20000] 0.001s
  [3500/20000] 0.509s
  [4000/20000] 0.627s
  [4500/20000] 0.021s
  [5000/20000] 0.090s
  [5500/20000] 0.706s
  [KILL error] 1317 (70100): Query execution was interrupted
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [TIMEOUT] Query killed after 10s
  [6000/20000] 0.859s
  [TIMEOUT] Query killed after 10s
  [KILL error] 1317 (70100): Query execution was interrupted
  [KILL error] 1317 (70100): Query execution was interrupted


In [11]:
df_metrics  = pd.DataFrame(metrics)

In [14]:
output_path = r"A:\pc\Desktop\Code\data science\sadop\Data\slow_query_metrics.csv"
df_metrics.to_csv(output_path, index=False)

timeouts = df_metrics["timed_out"].sum()
print(f"\n✅ Done!  {len(df_metrics)} rows saved → {output_path}")
print(f"   Timed-out queries : {timeouts} / {NUM_QUERIES}")



✅ Done!  20000 rows saved → A:\pc\Desktop\Code\data science\sadop\Data\slow_query_metrics.csv
   Timed-out queries : 25 / 20000
